# 训练与评测

阶段 1 把 Qwen3-Omni 的音频塔、新建的 projector 和 Qwen3.5-0.8B 的语言模型、视觉塔组合成了完整模型。这时 projector 的参数是随机初始化的，语言模型收到的是没有意义的向量，模型还听不懂音频。本教程用上一个教程生成的训练清单训练 projector：音频塔、视觉塔和语言模型的参数保持不动，只调整 projector，让它把音频塔输出的特征转换成语言模型能理解的输入。训练完成后，在 FLEURS 真人录音上评测转写的准确率。

```text
音频 ─▶ 音频塔（冻结）─▶ projector（训练）─▶ 语言模型（冻结）─▶ 转写文字
```

训练代码在 `pipeline/train/`，评测代码在 `pipeline/eval/`。训练和评测都在 GPU 上运行，分别使用工作目录中的 `envs/mllm-train` 和 `envs/mllm-eval` 环境，本 notebook 以子进程调用它们。完整规模的训练和评测已经用命令行脚本跑完，本 notebook 演示每一步的操作，再读取完整运行的结果。

## 0. 准备路径

路径的约定与上一个教程相同：工作目录由环境变量 `WORK_DIR` 指定，数据集目录由 `DATASET` 指定。`model_dir` 是阶段 1 保存的完整模型目录。`train_runs` 是完整规模的三次训练的输出目录，`eval_dir` 是它们的评测结果目录，后面会解释这三次训练的区别。

In [1]:
import json
import os
import shutil
import subprocess
from collections import Counter
from pathlib import Path

repo_dir = Path.cwd()
work_dir = Path(os.environ.get("WORK_DIR", repo_dir.parent / "qwen3.5-asr-work"))
dataset_dir = Path(os.environ["DATASET"])
audio_dir = dataset_dir / "audio" / "qwen3.5-asr"
model_dir = repo_dir / "qwen3.5-0.8b-mllm"
demo_dir = work_dir / "tmp" / "tutorial"
demo_dir.mkdir(parents = True, exist_ok = True)
train_python = str(work_dir / "envs" / "mllm-train" / "bin" / "python")
eval_python = str(work_dir / "envs" / "mllm-eval" / "bin" / "python")
train_runs = {name: work_dir / "data" / "T14" / name for name in ["v1a", "v1b", "v1b-20k"]}
eval_dir = work_dir / "data" / "T16"


def read_jsonl(path):
    with open(file = path, mode = "r", encoding = "utf-8") as handle:
        return [json.loads(s = line) for line in handle if line.strip()]

## 1. 训练样本长什么样

### 1.1 对话格式

训练和评测使用同一种输入：用户消息是一段音频加一句固定指令 `Transcribe the audio.`，不告诉模型是什么语言。这条消息用模型自带的 chat template 展开：`add_generation_prompt = True` 在末尾加上助手回合的开头，`enable_thinking = False` 让助手回合以一个空的思考块开始。模型要从这个位置开始写出转写文字，最后输出 `<|im_end|>` 表示说完了。下面是 `pipeline/train/data.py` 中构造提示的两个函数：

```python
INSTRUCTION = "Transcribe the audio."


def build_messages(audio_path):
    """按 CONVENTIONS 的对话格式构造用户消息：一段音频加一句固定指令，不给语言提示。"""
    return [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio": str(audio_path)},
                {"type": "text", "text": INSTRUCTION},
            ],
        },
    ]


def build_prompt(processor, audio_path):
    """展开 chat template，结尾是助手回合开头和空的思考块 <think>\n\n</think>\n\n。"""
    return processor.apply_chat_template(
        conversation = build_messages(audio_path = audio_path),
        tokenize = False,
        add_generation_prompt = True,
        enable_thinking = False,
    )
```

### 1.2 只对答案计算损失

训练时把提示和答案（目标文本加 `<|im_end|>`）拼成一条序列送进模型。模型在每个位置预测下一个 token，但提示部分不需要学，只有答案部分计算损失。`pipeline/train/data.py` 的 `TrainCollator` 用标签 `labels` 表示这一点：提示部分的标签填 -100，计算损失时跳过；答案部分的标签就是答案自己的 token 编号。

```python
answer_ids = list(ids) + [self.im_end_id]
sequences.append({
    "input_ids": prompt_ids + answer_ids,
    "labels": [IGNORE_INDEX] * len(prompt_ids) + answer_ids,
    "mm_token_type_ids": prompt_types + [0] * len(answer_ids),
})
```

`pipeline.train.show_batch` 把一条样本展开后的完整序列打印出来，计算损失的位置标上 `[loss]`。下面用上一个教程演示合成的一条英语样本。

In [2]:
demo_manifest = demo_dir / "demo_synth" / "manifest.jsonl"
result = subprocess.run(
    args = [
        train_python, "-m", "pipeline.train.show_batch",
        "--model-dir", str(model_dir),
        "--manifest", str(demo_manifest),
        "--target", "original",
        "--count", "1",
    ],
    check = True,
    capture_output = True,
    text = True,
)
print(result.stdout)

批次形状：input_ids (1, 194)，input_features (1, 128, 1074)

== train-en-hplt-000007（en，10.747 秒）
特征帧数 1074，音频塔规则给出的特征数 140，序列中的 <|audio_pad|> 140 个，有效长度 194，填充 0
目标文本：In the workplace we need to adapt to meet this expectation along with the idea that our users not only want to access data when they want but from whatever device they choose.
计损失的 token 35 个，解码为："In the workplace we need to adapt to meet this expectation along with the idea that our users not only want to access data when they want but from whatever device they choose.<|im_end|>"
第一个计损失的 token 之前的 6 个 token："assistant\n<think>\n\n</think>\n\n"
序列："<|im_start|>" "user" "\n" "<|audio_start|>" <|audio_pad|>×140 "<|audio_end|>" "Trans" "cribe" " the" " audio" "." "<|im_end|>" "\n" "<|im_start|>" "assistant" "\n" "<think>" "\n\n" "</think>" "\n\n" [loss]"In" [loss]" the" [loss]" workplace" [loss]" we" [loss]" need" [loss]" to" [loss]" adapt" [loss]" to" [loss]" meet" [loss]" this" [loss]" expectation" [loss]" along" [loss]" with" 

这条音频长 10.7 秒。processor 先把波形转成 1074 帧特征（每 10 毫秒一帧），音频塔把它压缩成 140 个特征（约每 77 毫秒一个），提示里因此预留了 140 个 `<|audio_pad|>` 占位，模型运行时用 projector 的输出替换这些占位。占位后面是指令、助手回合开头和空的思考块，这些都没有 `[loss]` 标记；只有答案的 34 个 token 和结尾的 `<|im_end|>` 计算损失。

### 1.3 训练目标：朗读稿还是原文

训练清单里每个样本同时有原文（`original`）和朗读稿（`script`），训练时用 `--target` 选择让模型学着输出哪一种：`script` 输出念出来的写法，`original` 输出书面写法。中文原文中可能有繁体，作为训练目标时转成简体，与朗读稿一致。大部分句子没有数字和符号，两种写法完全相同。下面从合并后的训练清单中统计两种写法不同的比例，并各看一条中文和英语的例子。

In [3]:
train_manifest = audio_dir / "v1" / "train" / "manifest-r1.jsonl"
rows = read_jsonl(path = train_manifest)
different = [row for row in rows if row["original"] != row["script"]]
print(f"训练清单 {len(rows)} 条，原文与朗读稿不同的 {len(different)} 条（{len(different) / len(rows):.1%}）")
for lang in ["zh", "en"]:
    row = next(row for row in different if row["lang"] == lang and any(ch.isdigit() for ch in row["original"]))
    print(f"{lang}  原文：{row['original']}")
    print(f"    朗读稿：{row['script']}")

训练清单 43778 条，原文与朗读稿不同的 4733 条（10.8%）
zh  原文：1、《老酒馆》中小棉袄结局是牺牲了。
    朗读稿：一、《老酒馆》中小棉袄结局是牺牲了。
en  原文：The experimental design consisted of a pre-test, a post-test, and a training procedure. One hundred and eighty children aged 6 to 8 years old were divided into three groups: task group, control group and training group.
    朗读稿：The experimental design consisted of a pre-test, a post-test, and a training procedure. One hundred and eighty children aged six to eight years old were divided into three groups: task group, control group and training group.


## 2. 只训练 projector

`pipeline/train/model.py` 的 `load_model` 加载阶段 1 的完整模型，只让名字以 `model.audio_projector.` 开头的参数参与训练。冻结的部分用 bfloat16 加载，节省显存；projector 转成 float32，优化器更新参数时保持足够的精度。

```python
model = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path = str(model_dir),
    trust_remote_code = True,
    dtype = dtype,
    attn_implementation = attn_implementation,
)
model.model.audio_projector.float()
for name, parameter in model.named_parameters():
    parameter.requires_grad_(requires_grad = name.startswith(PROJECTOR_PREFIX))
```

下面在 CPU 上加载完整模型，按模块统计参数量，看要训练的部分有多大。

In [4]:
import torch
from transformers import AutoModelForImageTextToText

model = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path = str(model_dir),
    trust_remote_code = True,
    dtype = torch.bfloat16,
)
parts = Counter()
for name, parameter in model.named_parameters():
    if name.startswith("model.audio_projector."):
        parts["projector"] += parameter.numel()
    elif name.startswith("model.audio_tower."):
        parts["音频塔"] += parameter.numel()
    elif name.startswith("model.visual."):
        parts["视觉塔"] += parameter.numel()
    else:
        parts["语言模型"] += parameter.numel()
total = sum(parts.values())
for part, count in parts.most_common():
    print(f"{part:6} {count / 1e6:8.1f} M  {count / total:6.2%}")
del model

语言模型      752.4 M  50.02%
音频塔       647.9 M  43.08%
视觉塔       100.6 M   6.69%
projector      3.1 M   0.21%


整个模型约 15 亿个参数，要训练的 projector 只有约 315 万个，占 0.21%。

训练的损失是交叉熵：对答案里的每个 token，模型给出下一个 token 是什么的概率分布，损失是正确 token 的概率取负对数，再对所有答案 token 求平均。损失越小，模型给正确答案的概率越高。可以用 e 的负损失次方粗略理解：损失 3.6 相当于平均给正确 token 约 3% 的概率，损失 1.25 相当于约 29%，损失 0.66 相当于约 52%。

## 3. 训练设置

`pipeline/train/run.sh` 用实验 v1 的超参数运行 `pipeline.train.train`：

| 参数 | 设置 | 含义 |
| --- | --- | --- |
| `--batch-size` | 16 | 每步用 16 条音频计算一次梯度 |
| `--max-batch-seconds` | 360 | 每批「条数 × 批内最长音频时长」不超过 360 秒；遇到长音频时一批少于 16 条，防止显存超过 32 GB |
| `--max-duration-s` | 30 | 超过 30 秒的音频不参加训练 |
| `--lr` | 3e-3 | 峰值学习率 |
| `--warmup-steps` | 500 | 前 500 步学习率从 0 线性升到峰值 |
| `--min-lr-ratio` | 0.1 | 预热之后按余弦曲线下降，最后一步降到峰值的 0.1 倍 |
| `--epochs` | 2 | 训练 2 轮，即每条音频用 2 次；也可以用 `--max-steps` 直接指定步数 |
| `--eval-every`、`--save-every` | 500 | 每 500 步在 FLEURS dev（真人录音，每种语言 50 句）上计算一次验证损失，并保存检查点 |

分批时先按时长把相近的样本放在一起（`length_grouped_batches`），一批内的音频长短接近，需要补齐的空白少；批次之间再打乱顺序。训练中途可以随时停止，重新运行同一条命令会从最近的检查点继续。

### 3.1 短训练演示

下面用合并后的训练清单跑 300 步：预热改成 50 步，每 100 步验证一次。日志每 50 步一行，给出速度、训练损失、学习率和显存；验证行给出 FLEURS dev 上的损失。模型加载和三次验证加起来，整个过程约 2 分钟。

In [5]:
demo_train = demo_dir / "train_demo"
shutil.rmtree(path = demo_train, ignore_errors = True)
result = subprocess.run(
    args = [
        "pipeline/train/run.sh", str(train_manifest), "original", str(demo_train),
        "--max-steps", "300",
        "--warmup-steps", "50",
        "--eval-every", "100",
        "--save-every", "100",
        "--log-every", "50",
    ],
    check = True,
    stdout = subprocess.PIPE,
    stderr = subprocess.STDOUT,
    text = True,
)
for line in result.stdout.splitlines():
    if line.startswith("[步") or line.startswith('{"eval_step"'):
        print(line)

[步 50/300 16.7%] 1.29 it/s | 20.3 样本/s | 134 音频h/h | loss 4.2927 | lr 3.00e-03 | 显存 17.4 GB | 已用 0:00:39 | 剩余 0:03:14
[步 100/300 33.3%] 6.33 it/s | 101.0 样本/s | 732 音频h/h | loss 3.9775 | lr 2.74e-03 | 显存 17.2 GB | 已用 0:00:47 | 剩余 0:01:33
{"eval_step": 100, "loss": 3.8064, "lang_mean_loss": 3.7521, "token_acc": 0.3403, "seconds": 19.0}
[步 150/300 50.0%] 6.39 it/s | 100.4 样本/s | 644 音频h/h | loss 3.8706 | lr 2.07e-03 | 显存 17.2 GB | 已用 0:01:13 | 剩余 0:01:13
[步 200/300 66.7%] 6.95 it/s | 106.9 样本/s | 774 音频h/h | loss 3.7910 | lr 1.23e-03 | 显存 18.1 GB | 已用 0:01:21 | 剩余 0:00:40
{"eval_step": 200, "loss": 3.7249, "lang_mean_loss": 3.6773, "token_acc": 0.3425, "seconds": 6.7}
[步 250/300 83.3%] 6.15 it/s | 96.4 样本/s | 604 音频h/h | loss 3.7472 | lr 5.58e-04 | 显存 16.5 GB | 已用 0:01:35 | 剩余 0:00:19
[步 300/300 100.0%] 7.30 it/s | 115.0 样本/s | 733 音频h/h | loss 3.7641 | lr 3.00e-04 | 显存 16.3 GB | 已用 0:01:42 | 剩余 0:00:00
{"eval_step": 300, "loss": 3.7002, "lang_mean_loss": 3.6588, "token_acc": 0.3446, "se

300 步里，训练损失从 4.29 降到 3.76，验证损失从第 100 步的 3.75 降到第 300 步的 3.66，答案 token 的预测正确率（`token_acc`）停在 34% 左右。这就是完整训练开头的平台期，在完整训练中要到第 1,500–2,500 步才结束，后面第 4 节会看到。训练速度约每秒 6–7 步、每秒约 100 条音频，相当于每小时处理约 700 小时音频，显存约 17 GB。

## 4. 完整训练的过程

完整规模训练了三个版本，训练数据都是第一轮合成的 43,771 条、81.1 小时：

| 版本 | 训练目标 | 训练步数 | 用时 |
| --- | --- | ---: | ---: |
| v1a | 朗读稿（`script`） | 5,556 步，2 轮 | 15 分钟 |
| v1b | 原文（`original`） | 5,556 步，2 轮 | 15 分钟 |
| v1b-20k | 原文（`original`） | 20,000 步，约 7.2 轮，预热 1,000 步 | 51 分钟 |

v1a 和 v1b 比较两种训练目标。v1b-20k 用来看多训练几轮验证损失会不会继续下降。每次训练的输出目录里，`eval_log.jsonl` 记录每次验证的结果，`train_log.jsonl` 记录训练日志。下面列出三次训练在若干步数时的验证损失（先按语言算损失，再对 18 种语言取平均），以及 v1b-20k 同期的训练损失。

In [6]:
def eval_losses(run):
    return {row["step"]: row["overall"]["lang_mean_loss"] for row in read_jsonl(path = run / "eval_log.jsonl")}


losses = {name: eval_losses(run = run) for name, run in train_runs.items()}
train_loss = {row["step"]: row["loss"] for row in read_jsonl(path = train_runs["v1b-20k"] / "train_log.jsonl")}
print("步数     v1a    v1b  v1b-20k  v1b-20k 训练损失")
for step in [500, 1500, 2000, 2500, 3000, 4000, 5000, 5556, 8000, 10000, 12000, 14000, 16000, 18000, 19500, 20000]:
    cells = [f"{losses[name][step]:6.3f}" if step in losses[name] else "      " for name in train_runs]
    print(f"{step:6}  " + "  ".join(cells) + (f"  {train_loss[step]:8.3f}" if step in train_loss else ""))

步数     v1a    v1b  v1b-20k  v1b-20k 训练损失
   500   3.616   3.605   3.617     3.704
  1500   3.477   3.413   3.520     3.467
  2000   1.836   1.800   3.419     3.387
  2500   1.587   1.564   1.853     1.505
  3000   1.493   1.450   1.672     1.314
  4000   1.395   1.339   1.531     1.145
  5000   1.337   1.277   1.496     1.110
  5556   1.318   1.252        
  8000                   1.401     0.957
 10000                   1.350     0.909
 12000                   1.303     0.812
 14000                   1.276     0.761
 16000                   1.245     0.735
 18000                   1.241     0.690
 19500                   1.221     0.651
 20000                   1.233     0.651


三次训练都有同样的形状：前 1,500 步左右验证损失停在 3.4–3.6，几乎不动，随后在几百步内骤降到 1.8 左右，再缓慢下降。v1b-20k 的预热更长，骤降也晚一些，出现在第 2,000 到 2,500 步之间。

v1a 和 v1b 训练 2 轮，结束时验证损失分别是 1.318 和 1.252，而且一直在下降。v1b-20k 训练到第 16,000 步（约 5.8 轮）以后，每 500 步的变化不超过 ±0.017，有升有降，基本稳定在 1.22–1.24，最低是第 19,500 步的 1.221，只比 2 轮的 v1b 低 0.03。同一时期训练损失仍从 0.92 降到 0.66：模型越来越适应训练用的合成音频，对真人录音的帮助却很小。也就是说，在这 81 小时数据上多训练几轮已经没有多少收益，继续提高需要更多、更多样的数据。

## 5. 用训练好的 projector 转写

每次训练保存了多个检查点，评测时用验证损失最低的一个。`pipeline/train/best_checkpoint.py` 读取每个检查点的 `projector.json`，取其中验证损失最低者：

In [7]:
from pipeline.train.best_checkpoint import best_checkpoint

best = {}
for name, run in train_runs.items():
    loss, directory = best_checkpoint(output_dir = run)
    best[name] = directory
    print(f"{name:8} {directory.name}  验证损失 {loss:.3f}")

v1a      checkpoint-0005556  验证损失 1.318
v1b      checkpoint-0005556  验证损失 1.252
v1b-20k  checkpoint-0019500  验证损失 1.221


检查点里只保存 projector 的参数（`projector.safetensors`，13 MB）。评测脚本 `pipeline.eval.run_eval` 先加载阶段 1 的完整模型，再把 projector 的参数换成检查点里的值，然后用与训练相同的提示做贪心解码：每一步取概率最高的 token，直到输出 `<|im_end|>`。下面用 v1b-20k 的检查点，转写 FLEURS 测试集中中文、英语、日语、西班牙语各 4 句。

In [8]:
demo_eval = demo_dir / "eval_demo"
shutil.rmtree(path = demo_eval, ignore_errors = True)
subprocess.run(
    args = [
        eval_python, "-m", "pipeline.eval.run_eval",
        "--model-dir", str(model_dir),
        "--projector", str(best["v1b-20k"] / "projector.safetensors"),
        "--manifest-dir", str(audio_dir / "fleurs"),
        "--langs", "zh", "en", "ja", "es",
        "--limit", "4",
        "--batch-size", "16",
        "--output-dir", str(demo_eval),
    ],
    check = True,
    capture_output = True,
)
for row in read_jsonl(path = demo_eval / "hypotheses.jsonl"):
    print(f"{row['lang']}  错误率 {row['numeric']['rate']:.0%}")
    print(f"    正确答案：{row['reference']}")
    print(f"    模型转写：{row['hypothesis']}")

zh  错误率 48%
    正确答案：钙、钾等元素属于金属，银和金等元素当然也是金属。
    模型转写：骨甲等元素属于金属，另外也是金属。
zh  错误率 9%
    正确答案：“这并不是告别。这是一个篇章的结束，也是新篇章的开始。”
    模型转写：这并非告别，这是一个篇章的结束，也是新篇章的开始。
zh  错误率 11%
    正确答案：桥下垂直净空 15 米。该项目于 2011 年 8 月完工，但直到 2017 年 3 月才开始通车。
    模型转写：桥下垂直进库（25 米）该项目于2013年8月完工，但直到2017年3月才开始通车。
zh  错误率 27%
    正确答案："适当使用博客，"可以使学生变得更善于分析和进行思辨；通过积极回应网络材料，学生们可以在他人文章的上下文语境中找到自己的立场，并能够针对特定问题提出自己的观点（Oravec, 2002）。"
    模型转写：适当使用博客可以使得学生变得更善于分析和进行四辩通过积极回忆网络材料，学生们可以在他人的文章中对自己的立场，并能够针对特定问题提出自己的观点。
en  错误率 27%
    正确答案：The cabbage juice changes color depending on how acidic or basic (alkaline) the chemical is.
    模型转写：The cabbage crushes color depending on how acidic, basic, or alkaline the chemical is.
en  错误率 19%
    正确答案：All nouns, alongside the word Sie for you, always begin with a capital letter, even in the middle of a sentence.
    模型转写：All names accordingly the world say for you always begin with a capital letter, even in the middle of a sentence.
en  错误率 5%
    正确答案：However, due to the slow communica

英语和西班牙语多数句子的错误率在 5%–27%，错误多是读音相近的词：「cabbage juice」写成「cabbage crushes」，「funeral」写成「funnel」，「Sintra」写成「Cinsra」。中文第二句只有「并不是」写成「并非」，另外标点不同；第三句把「净空 15 米」听成「进库 25 米」，把「2011」听成「2013」。日语的错误更多，模型能写出读音相近的内容，汉字却常常选错，例如把「公用語」写成「公演」。日语第二句陷入了重复，「弓矢に弓の矢が納め」一直输出到长度上限，错误率达到 305%。这种情况叫解码失控，第 6 节会统计它对结果的影响。

这里每种语言只看了 4 句，下面看完整测试集上的结果。

## 6. 评测

### 6.1 评测集和指标

评测用了三套测试集：

1. FLEURS 测试集：18 种语言的真人朗读录音，共 13,949 句，是最重要的一套，反映模型听真人说话的水平。
2. 留出音色合成测试集：用训练时从未用过的音色合成，454 句，检验模型对没听过的声音是否有效。
3. 训练音色合成测试集：句子与第 2 套相同，换成训练用过的音色，506 句，与第 2 套对比。

错误率的算法：把模型的转写与正确答案逐词对齐，统计多出、漏掉、写错的词数，除以正确答案的词数。中文、日语、韩语、泰语没有词间空格，按字计算，叫 CER（字错误率）；其余语言按词计算，叫 WER（词错误率）。多出的词也算错误，所以错误率可以超过 100%。打分前统一大小写、删除标点，并把「二十」和「20」这类数词统一成数值（数词统一），这样只是数字写法不同时不算错；用朗读稿训练的 v1a 输出「二十」，用原文训练的 v1b 输出「20」，统一之后才能公平比较。「各语言平均」指先算每种语言的错误率，再对 18 种语言取简单平均，每种语言的权重相同。

### 6.2 FLEURS 测试集

完整规模的评测用 `pipeline/eval/run.sh` 运行，转写一遍 FLEURS 测试集约 27 分钟，结果在 `eval_dir` 中。`pipeline.eval.report` 把多次评测汇总成表格，下面运行它，先看 FLEURS 测试集上每种语言的错误率。

In [9]:
report_dir = demo_dir / "report"
runner_python = str(work_dir / "envs" / "runner" / "bin" / "python")
subprocess.run(
    args = [
        runner_python, "-m", "pipeline.eval.report",
        "--out-dir", str(report_dir),
        "--fleurs", *[f"{name}={eval_dir / (name + '-fleurs')}" for name in train_runs],
        "--synth", *[
            f"{name}:{test_set}={eval_dir / f'{name}-{test_set}'}@{eval_dir / f'{test_set}-manifest'}"
            for name in ["v1a", "v1b"]
            for test_set in ["holdout", "trainvoice"]
        ],
    ],
    check = True,
    capture_output = True,
)
tables = json.loads(s = (report_dir / "tables.json").read_text(encoding = "utf-8"))
print("FLEURS 测试集，数词统一后的错误率（%）")
print("语言  指标  句数    v1a    v1b  v1b-20k")
for lang, item in tables["fleurs"].items():
    metric = "CER" if lang in ("zh", "ja", "ko", "th") else "WER"
    print(f"{lang:4}  {metric}  {item['v1a']['n']:5}  " + "  ".join(f"{100 * item[name]['numeric']:6.1f}" for name in train_runs))
print("各语言平均          " + "  ".join(f"{100 * tables['fleurs_mean'][name]['numeric']:6.1f}" for name in train_runs))

FLEURS 测试集，数词统一后的错误率（%）
语言  指标  句数    v1a    v1b  v1b-20k
zh    CER    945    58.0    54.4    40.8
en    WER    647    25.1    23.0    29.5
ja    CER    650   142.0   118.1   111.7
ko    CER    382    49.1    52.9    42.8
de    WER    862    47.1    39.5    37.0
fr    WER    676    45.9    44.1    43.0
ru    WER    775    77.8    80.8    80.7
pt    WER    919    29.5    25.4    36.2
es    WER    908    26.4    26.5    21.9
it    WER    865    35.4    26.6    32.8
id    WER    687    81.6    74.4    77.1
ms    WER    749    88.2    76.8    68.2
th    CER   1021    58.0    60.2    64.7
vi    WER    857   184.7   171.9   149.2
fil   WER    964   110.0    93.5    95.6
ar    WER    428    80.1    80.1    82.1
tr    WER    743    76.2    70.1    78.9
fa    WER    871   155.9   139.5   114.4
各语言平均            76.2    69.9    67.0


三个版本的平均错误率分别是 76.2%、69.9% 和 67.0%。英语、西班牙语、葡萄牙语、意大利语最好，在 20%–36% 之间；越南语、波斯语、日语超过了 100%。错误率超过 100% 说明模型输出了比正确答案多得多的内容，下面找一句日语看看。

In [10]:
ja_rows = [row for row in read_jsonl(path = eval_dir / "v1b-20k-fleurs" / "hypotheses.jsonl") if row["lang"] == "ja"]
row = max(ja_rows, key = lambda row: len(row["hypothesis"]) / len(row["reference"]))
print("正确答案：", row["reference"])
print("模型转写：", row["hypothesis"][:150], "……")
print(f"转写 {len(row['hypothesis'])} 个字符，正确答案 {len(row['reference'])} 个字符")

正确答案： 地域的・季節的な厳しい気象現象としては、吹雪、吹雪、暴氷、砂嵐などがあります。
模型转写： 政治的、経済的、社会的な基本規範としては、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、政府、 ……
转写 321 个字符，正确答案 39 个字符


模型开头还在转写，随后陷入重复，同一段话一直输出到长度上限才停，这叫解码失控。贪心解码每一步只取概率最高的 token，一旦进入重复的循环就很难跳出。把输出字符数超过正确答案 2 倍再加 10 的句子算作失控，下面统计每种语言失控句子的比例，以及去掉这些句子后的错误率。

In [11]:
print("语言   失控句子比例（%）       去掉失控后的错误率（%）")
print("        v1a   v1b  v1b-20k      v1a    v1b  v1b-20k")
for lang, item in tables["fleurs"].items():
    shares = "  ".join(f"{100 * item[name]['runaway']:4.1f}" for name in train_runs)
    rates = "  ".join(f"{100 * item[name]['numeric_no_runaway']:5.1f}" for name in train_runs)
    print(f"{lang:4}  {shares}       {rates}")
means = tables["fleurs_mean"]
print("全部  " + "  ".join(f"{100 * means[name]['runaway']:4.1f}" for name in train_runs) + "       " + "  ".join(f"{100 * means[name]['numeric_no_runaway']:5.1f}" for name in train_runs))

语言   失控句子比例（%）       去掉失控后的错误率（%）
        v1a   v1b  v1b-20k      v1a    v1b  v1b-20k
zh     5.4   6.1   3.2        28.6   28.5   26.6
en     1.9   1.4   1.7        15.6   15.0   17.6
ja    19.5  13.8  13.2        60.3   59.7   57.6
ko     1.6   3.1   2.4        37.2   39.4   32.1
de     2.4   1.9   1.9        30.0   30.2   30.4
fr     2.8   3.0   2.2        33.9   33.7   33.3
ru     4.3   4.4   4.5        61.4   63.4   62.0
pt     1.1   0.7   2.0        22.9   22.8   23.8
es     2.0   1.7   0.9        17.1   17.4   18.1
it     2.0   1.0   1.7        23.7   24.4   23.3
id     5.2   5.4   4.8        41.7   42.1   43.7
ms     7.5   4.7   3.9        52.5   51.7   52.0
th     4.9   5.4   6.4        45.7   46.5   46.1
vi    25.1  22.2  19.7        62.2   63.5   62.7
fil   10.5   8.0   8.4        56.9   55.4   55.0
ar     1.9   3.5   4.7        68.5   68.4   66.0
tr     3.5   2.4   5.0        57.6   58.0   56.7
fa    15.0  12.7   9.2        94.1   87.7   80.3
全部   6.7   5.8   5.4        45.0

失控集中在越南语、日语、波斯语、菲律宾语，英语、西班牙语、葡萄牙语只有 1%–2%。去掉失控句子后，三个版本的平均错误率是 45.0%、44.9% 和 43.7%，几乎相同：训练目标用朗读稿还是原文，对模型正常输出时的准确率几乎没有影响，v1b 的平均错误率更低，主要因为失控的句子少一些；训练 7 轮的 v1b-20k 只比 2 轮好 1.2 个百分点。去掉失控后，v1b-20k 在英语、西班牙语上约 18%，葡萄牙语、意大利语、中文约 23%–27%。没有失控的错误大多是读音相近、用字或分词不对，说明 projector 已经把语音内容传给了语言模型。

### 6.3 没听过的音色

留出音色和训练音色两套合成测试集用同一批句子，各自只保留核对通过的句子。只看两套都通过的同一批句子，比较模型在两种音色上的表现，并按合成方式分开：内置音色是 TTS 自带的说话人，克隆是 TTS 模仿一段参考录音。

In [12]:
common = next(value for key, value in tables["synth"].items() if key.startswith("两份测试集"))
print("逐句错误率的平均（%），括号内为句数")
print("合成方式   v1a 留出音色   v1a 训练音色   v1b 留出音色   v1b 训练音色")
for mode, item in common.items():
    cells = [f"{100 * item[key]['mean_rate']:5.1f}（{item[key]['n']:3}）" for key in ["v1a-holdout", "v1a-trainvoice", "v1b-holdout", "v1b-trainvoice"]]
    print(f"{mode:8}   " + "    ".join(cells))

逐句错误率的平均（%），括号内为句数
合成方式   v1a 留出音色   v1a 训练音色   v1b 留出音色   v1b 训练音色
builtin     27.3（217）     25.9（231）     30.9（217）     26.6（231）
clone       43.9（115）     47.5（101）     44.5（115）     40.7（101）


换成训练时没出现过的音色，错误率只高 0 到 4 个百分点，模型没有死记训练用的声音。克隆合成的音频明显更难识别，不论音色有没有在训练中出现过，错误率都在 40%–48%，内置音色在 26%–31%。本实验克隆用的参考录音本身也是 TTS 合成的，克隆音频相当于「合成的合成」，这可能是它更难识别、合成时核对通过率也更低的原因。

## 7. 结论

这条路线能产出可用的训练数据：只训练约 315 万个参数的 projector，模型就学会了从音频转写文字，英语、西班牙语、葡萄牙语、意大利语在真人录音上的词错误率约 20%–30%。瓶颈在数据量和多样性：81 小时分到 18 种语言，每种只有 4–5 小时，验证损失在约 6 轮后就不再下降；合成音频与真人录音之间也还有差距。后续计划加入有开放许可的真人录音，用 TTS 补足缺少真人录音的语言，并用真人录音作为声音克隆的参考。完整的数字和分析见 `results/v1-zh.md`。